# DataPulse — Reusable Bronze Ingestion (source-faithful)

This notebook ingests one configured dataset from ADLS to a Bronze Delta table and preserves source columns as much as possible.

The DataPulse reference PDFs document proposed field names and types. For source-faithful Bronze, avoid inferring numeric types for identifiers; apply validated casts in Silver. The real manifest schema and file contracts must be connected/confirmed before production.

In [0]:
dbutils.widgets.text("source_name", "google_ads")
dbutils.widgets.text("dataset_name", "campaigns")
dbutils.widgets.text("market", "IN")
dbutils.widgets.text("processing_date", "2026-10-28")
dbutils.widgets.text("source_path", "")
dbutils.widgets.text("file_format", "json")
dbutils.widgets.text("target_catalog_schema", "main.datapulse_bronze")
dbutils.widgets.text("target_table", "")
dbutils.widgets.text("header", "true")
dbutils.widgets.text("delimiter", ",")
dbutils.widgets.text("write_mode", "append")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("batch_id", "")

source_name = dbutils.widgets.get("source_name").strip().lower()
dataset_name = dbutils.widgets.get("dataset_name").strip().lower()
market = dbutils.widgets.get("market").strip().upper()
processing_date = dbutils.widgets.get("processing_date").strip()
source_path = dbutils.widgets.get("source_path").strip()
file_format = dbutils.widgets.get("file_format").strip().lower()
target_catalog_schema = dbutils.widgets.get("target_catalog_schema").strip()
target_table = dbutils.widgets.get("target_table").strip()
header = dbutils.widgets.get("header").strip().lower() == "true"
delimiter = dbutils.widgets.get("delimiter")
write_mode = dbutils.widgets.get("write_mode").strip().lower()
run_id = dbutils.widgets.get("run_id").strip()
batch_id = dbutils.widgets.get("batch_id").strip()

In [0]:
from datetime import datetime, timezone
from pyspark.sql import functions as F
import re

DATASETS = {
    "google_ads": {"campaigns","ad_groups","ads","keywords","campaign_daily_performance"},
    "amazon_ads": {"campaigns","ad_groups","product_ads","keywords","campaign_daily_performance"},
    "meta_ads": {"campaigns","ad_sets","ads","creatives","daily_insights"},
    "magento": {"customers","products","orders","order_items","order_refunds"},
    "sap": {"customers","materials","sales_order_items","billing_items","inventory_snapshots"},
}
if source_name not in DATASETS or dataset_name not in DATASETS[source_name]:
    raise ValueError(f"Unsupported source/dataset: {source_name}/{dataset_name}")
if not source_path:
    raise ValueError("source_path is required until the real manifest lookup is wired in.")
if file_format not in {"csv","json","parquet"}:
    raise ValueError("file_format must be csv, json, or parquet")
if write_mode not in {"append","overwrite"}:
    raise ValueError("write_mode must be append or overwrite")
if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", processing_date):
    raise ValueError("processing_date must use YYYY-MM-DD")

target_table = target_table or f"{target_catalog_schema}.bronze_{source_name}_{dataset_name}"
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*(\.[A-Za-z_][A-Za-z0-9_]*){1,2}", target_table):
    raise ValueError(f"Invalid target_table identifier: {target_table}")

In [0]:
reader = spark.read
if file_format == "csv":
    # Keep CSV values as strings in Bronze to protect IDs and preserve source representation.
    reader = reader.option("header", str(header).lower()).option("delimiter", delimiter).option("mode","PERMISSIVE").option("inferSchema","false")
elif file_format == "json":
    # Avoid inferring primitive numeric values for identifiers where supported by Spark.
    reader = reader.option("mode","PERMISSIVE").option("primitivesAsString","true")
df = reader.format(file_format).load(source_path)
if not df.columns:
    raise ValueError(f"No columns were read from {source_path}")

source_rows = df.count()
ingested_at = datetime.now(timezone.utc).isoformat()
df_bronze = (
    df.withColumn("_source_name", F.lit(source_name))
      .withColumn("_dataset_name", F.lit(dataset_name))
      .withColumn("_market", F.lit(market or None))
      .withColumn("_processing_date", F.to_date(F.lit(processing_date)))
      .withColumn("_source_file_path", F.input_file_name())
      .withColumn("_ingested_at_utc", F.lit(ingested_at).cast("timestamp"))
      .withColumn("_run_id", F.lit(run_id or None))
      .withColumn("_batch_id", F.lit(batch_id or None))
)

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_table.rsplit('.', 1)[0]}")
writer = df_bronze.write.format("delta").mode(write_mode)
if write_mode == "overwrite":
    writer = writer.option("overwriteSchema","true")
writer.saveAsTable(target_table)

print({
    "status":"SUCCESS","source_name":source_name,"dataset_name":dataset_name,
    "market":market or None,"processing_date":processing_date,
    "source_path":source_path,"target_table":target_table,
    "source_rows":source_rows,"run_id":run_id or None,"batch_id":batch_id or None
})

## Manifest integration and safe reruns
- Add manifest selection using the actual DataPulse manifest table/schema.
- Claim the selected file/batch before processing, using concurrency-safe logic.
- Write with MERGE or controlled ingestion-slice replacement so retries do not duplicate rows.
- Update manifest status to SUCCESS only after the Delta write succeeds; capture FAILED status and error details otherwise.
- Do not use append as the only duplicate-prevention mechanism.